# AI Loan Advisory Chatbot
## Phase 1: PDF Processing Pipeline

### Objective

This notebook is responsible for processing all official banking PDF documents.

It performs the following tasks:

- Read PDF files
- Extract text
- Clean extracted text
- Generate metadata
- Save processed outputs

The processed data generated in this notebook will be used in the subsequent stages:
- Intelligent Chunking
- Embedding Generation
- FAISS Vector Database
- Retrieval-Augmented Generation (RAG)

## Why PDF Processing?

Large Language Models cannot directly understand PDF files.

Before creating embeddings, every document must be converted into clean text.

This notebook converts banking PDFs into a standardized format that can later be used for semantic search and question answering.

Pipeline:

Official PDFs
      ↓
Text Extraction
      ↓
Cleaning
      ↓
Metadata
      ↓
Ready for Chunking

In [1]:
!pip install -q pymupdf pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 111.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 109.1 MB/s eta 0:00:00


In [2]:
import fitz
import pdfplumber
from pathlib import Path
import json
import re
import os
from datetime import datetime

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
PROJECT_ROOT = Path("/content/drive/MyDrive/AI_Loan_Advisory_Chatbot")

DATASET_DIR = PROJECT_ROOT / "dataset"

PROCESSED_DIR = PROJECT_ROOT / "processed"

CLEAN_TEXT_DIR = PROCESSED_DIR / "cleaned_text"

METADATA_DIR = PROCESSED_DIR / "metadata"

CHUNKS_DIR = PROCESSED_DIR / "chunks"

VECTOR_DB_DIR = PROJECT_ROOT / "vector_db"

MODELS_DIR = PROJECT_ROOT / "models"

OUTPUTS_DIR = PROJECT_ROOT / "outputs"

In [5]:
folders = [CLEAN_TEXT_DIR,METADATA_DIR,CHUNKS_DIR,VECTOR_DB_DIR,MODELS_DIR,OUTPUTS_DIR]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("Project structure is ready!")

Project structure is ready!


In [6]:
for folder in PROJECT_ROOT.iterdir():
  print(folder.name)

dataset
processed
vector_db
models
notebooks
streamlit_app
utils
outputs
presentation


In [7]:
pdf_files=sorted(DATASET_DIR.rglob("*.pdf"))
print(f"Total PDFs Found:{len(pdf_files)}\n")
for pdf in pdf_files:
  print(pdf.relative_to(DATASET_DIR))

Total PDFs Found:36

HDFC/Car Loan/HDFC_Car_Loan_MITC.pdf
HDFC/Car Loan/HDFC_Car_Loan_Overview.pdf
HDFC/Home Loan/HDFC_Home_Loan_MITC.pdf
HDFC/Home Loan/HDFC_Home_Loan_Overview.pdf
HDFC/Personal Loan/HDFC_Personal_Loan_Overview.pdf
SBI/Education Loan/SBI_EDUCATION_LOAN_Main_Page.pdf
SBI/Education Loan/SBI_EDUCATION_LOAN_Scholar_Loan..pdf
SBI/Education Loan/SBI_EDUCATION_LOAN_Student_Loan.pdf
SBI/Education Loan/SBI_Education_Loan_Interest_Rate.pdf
SBI/Education Loan/SBI_Education_MITC_Scholar_Loan.pdf
SBI/Education Loan/SBI_Educational_Loan_Faq's.pdf
SBI/Gold Loan/SBI_GOLD_LOAN_Product_Overview.pdf.pdf
SBI/Home Loan/SBI_Home_Loan_Agreement.pdf
SBI/Home Loan/SBI_Home_Loan_Application_Form.pdf
SBI/Home Loan/SBI_Home_Loan_Documents_Required.pdf
SBI/Home Loan/SBI_Home_Loan_EMI_Calculator.pdf
SBI/Home Loan/SBI_Home_Loan_Eligibility.pdf
SBI/Home Loan/SBI_Home_Loan_FAQs.pdf
SBI/Home Loan/SBI_Home_Loan_Features.pdf
SBI/Home Loan/SBI_Home_Loan_Interest_Rates.pdf
SBI/Home Loan/SBI_Home_Loan_Inter

In [8]:
sample_pdf=pdf_files[0]
print("Full path:")
print(sample_pdf)
print("\nFilename:")
print(sample_pdf.name)
print("\nParent Folder:")
print(sample_pdf.parent.name)
print("\nBank:")
print(sample_pdf.parts[-3])
print("\nLoan Type:")
print(sample_pdf.parts[-2])

Full path:
/content/drive/MyDrive/AI_Loan_Advisory_Chatbot/dataset/HDFC/Car Loan/HDFC_Car_Loan_MITC.pdf

Filename:
HDFC_Car_Loan_MITC.pdf

Parent Folder:
Car Loan

Bank:
HDFC

Loan Type:
Car Loan


In [9]:
def get_pdf_files(dataset_path):
  """Recursively discover all PDF files in the dataset.
  Args:dataset_path (Path):Root dataset directory.
  Returns:list:Sorted list of PDF file paths.
  """
  pdf_files=sorted(dataset_path.rglob("*.pdf"))
  print(f"Found {len(pdf_files)} PDF files.\n")
  return pdf_files

In [10]:
pdf_files=get_pdf_files(DATASET_DIR)
for pdf in pdf_files:
  print(pdf.relative_to(DATASET_DIR))

Found 36 PDF files.

HDFC/Car Loan/HDFC_Car_Loan_MITC.pdf
HDFC/Car Loan/HDFC_Car_Loan_Overview.pdf
HDFC/Home Loan/HDFC_Home_Loan_MITC.pdf
HDFC/Home Loan/HDFC_Home_Loan_Overview.pdf
HDFC/Personal Loan/HDFC_Personal_Loan_Overview.pdf
SBI/Education Loan/SBI_EDUCATION_LOAN_Main_Page.pdf
SBI/Education Loan/SBI_EDUCATION_LOAN_Scholar_Loan..pdf
SBI/Education Loan/SBI_EDUCATION_LOAN_Student_Loan.pdf
SBI/Education Loan/SBI_Education_Loan_Interest_Rate.pdf
SBI/Education Loan/SBI_Education_MITC_Scholar_Loan.pdf
SBI/Education Loan/SBI_Educational_Loan_Faq's.pdf
SBI/Gold Loan/SBI_GOLD_LOAN_Product_Overview.pdf.pdf
SBI/Home Loan/SBI_Home_Loan_Agreement.pdf
SBI/Home Loan/SBI_Home_Loan_Application_Form.pdf
SBI/Home Loan/SBI_Home_Loan_Documents_Required.pdf
SBI/Home Loan/SBI_Home_Loan_EMI_Calculator.pdf
SBI/Home Loan/SBI_Home_Loan_Eligibility.pdf
SBI/Home Loan/SBI_Home_Loan_FAQs.pdf
SBI/Home Loan/SBI_Home_Loan_Features.pdf
SBI/Home Loan/SBI_Home_Loan_Interest_Rates.pdf
SBI/Home Loan/SBI_Home_Loan_Inter

In [11]:
def extract_metadata_from_path(pdf_path):
  """Extract metadata from the folder structure.
  Example:
  dataset/SBI/Home Loan/Home_Loan_FAQ.pdf
  Returns:dict"""
  relative_path=pdf_path.relative_to(DATASET_DIR)
  parts=relative_path.parts
  metadata={"bank":parts[0],"loan type":parts[1],"filen_name":pdf_path.name,"file_path":str(relative_path)}
  return metadata

In [12]:
sample = pdf_files[0]

metadata = extract_metadata_from_path(sample)

print(metadata)

{'bank': 'HDFC', 'loan type': 'Car Loan', 'filen_name': 'HDFC_Car_Loan_MITC.pdf', 'file_path': 'HDFC/Car Loan/HDFC_Car_Loan_MITC.pdf'}


In [13]:
from datetime import datetime

def extract_metadata_from_path(pdf_path):

    relative_path = pdf_path.relative_to(DATASET_DIR)

    parts = relative_path.parts

    metadata = {
        "bank": parts[0],
        "loan_type": parts[1],
        "file_name": pdf_path.name,
        "file_path": str(relative_path),
        "extension": pdf_path.suffix,
        "processed_date": datetime.now().strftime("%Y-%m-%d")
    }

    return metadata

In [14]:
def extract_text_from_pdf(pdf_path):
  """Extract text from all pages of a PDF.
  Parameters
  ----------
  pdf_path:Path
  Path to the PDF file.
  Returns
  -------
  str
  Complete extracted text.
  """
  document=fitz.open(pdf_path)
  text=""
  for page in document:
    text+=page.get_text()
  document.close()
  return text

In [15]:
sample_pdf=pdf_files[0]
text=extract_text_from_pdf(sample_pdf)
print(text[:3000])

Unknown Car Loan Terms & Conditions
Terms and Conditions
New Car Loan Terms and Conditions
1. I agree to abide by the Bank's Terms and Conditions and rules in force and the changes thereto in
Terms and Conditions from time to time relating to my account as communicated and made available
on the Bank's website.
2. I agree that the opening and maintenance of the account is subject to rules and regulations
introduced or amended from time to time by the Reserve Bank of India.
3. I agree that the bank before opening any deposit account, will carry out a due diligence as required
under Know Your Customer guidelines of the bank. I would be required to submit necessary documents
or proofs, such as identity , address, photograph and any such information to meet with KYC, AML or
other statutory/regulatory requirements.
Further, after the account is opened, in compliance with the extant regulatory guidelines, I agree to
submit the above documents again at periodic intervals, as may be required by

In [16]:
for pdf in pdf_files:
  text=extract_text_from_pdf(pdf)
  print("="*60)
  print(pdf.name)
  print(f"Characters:{len(text)}")

HDFC_Car_Loan_MITC.pdf
Characters:19927
HDFC_Car_Loan_Overview.pdf
Characters:8933
HDFC_Home_Loan_MITC.pdf
Characters:1837
HDFC_Home_Loan_Overview.pdf
Characters:50926
HDFC_Personal_Loan_Overview.pdf
Characters:40877
SBI_EDUCATION_LOAN_Main_Page.pdf
Characters:4849
SBI_EDUCATION_LOAN_Scholar_Loan..pdf
Characters:12927
SBI_EDUCATION_LOAN_Student_Loan.pdf
Characters:14115
SBI_Education_Loan_Interest_Rate.pdf
Characters:8610
SBI_Education_MITC_Scholar_Loan.pdf
Characters:7993
SBI_Educational_Loan_Faq's.pdf
Characters:7322
SBI_GOLD_LOAN_Product_Overview.pdf.pdf
Characters:13517
SBI_Home_Loan_Agreement.pdf
Characters:11024
SBI_Home_Loan_Application_Form.pdf
Characters:18472
SBI_Home_Loan_Documents_Required.pdf
Characters:3118
SBI_Home_Loan_EMI_Calculator.pdf
Characters:7414
SBI_Home_Loan_Eligibility.pdf
Characters:1200
SBI_Home_Loan_FAQs.pdf
Characters:24195
SBI_Home_Loan_Features.pdf
Characters:1340
SBI_Home_Loan_Interest_Rates.pdf
Characters:2282
SBI_Home_Loan_Interest_Rates_and_Fees.pdf


In [17]:
def extract_text_from_pdf(pdf_path):
    """
    Extract text and page count from a PDF.
    """
    document = fitz.open(pdf_path)
    text = ""
    for page in document:
        text += page.get_text()
    page_count = len(document)
    document.close()
    return text, page_count

In [18]:
text,pages=extract_text_from_pdf(sample_pdf)
print(f"Pages:{pages}")
print(f"Characters:{len(text)}")

Pages:6
Characters:19927


In [19]:
def extract_text_from_pdf(pdf_path):
    document = fitz.open(pdf_path)

    try:
        extracted_text = ""

        for page in document:
            extracted_text += page.get_text()

        return {
            "text": extracted_text,
            "page_count": len(document),
            "character_count": len(extracted_text),
            "word_count": len(extracted_text.split())
        }

    finally:
        document.close()

In [20]:
sample = extract_text_from_pdf(pdf_files[0])

print(sample.keys())
print(sample["page_count"])
print(sample["character_count"])
print(sample["word_count"])

dict_keys(['text', 'page_count', 'character_count', 'word_count'])
6
19927
3277


In [21]:
print(sample["text"[:5000]])

Unknown Car Loan Terms & Conditions
Terms and Conditions
New Car Loan Terms and Conditions
1. I agree to abide by the Bank's Terms and Conditions and rules in force and the changes thereto in
Terms and Conditions from time to time relating to my account as communicated and made available
on the Bank's website.
2. I agree that the opening and maintenance of the account is subject to rules and regulations
introduced or amended from time to time by the Reserve Bank of India.
3. I agree that the bank before opening any deposit account, will carry out a due diligence as required
under Know Your Customer guidelines of the bank. I would be required to submit necessary documents
or proofs, such as identity , address, photograph and any such information to meet with KYC, AML or
other statutory/regulatory requirements.
Further, after the account is opened, in compliance with the extant regulatory guidelines, I agree to
submit the above documents again at periodic intervals, as may be required by

In [22]:
def clean_text(text):
  """Clean extracted PDF text while preserving important banking information."""
  text=text.replace("ï¿½", "'")
  text=text.replace("\r","\n")
  text=text.replace("\t"," ")
  text=re.sub(r" +"," ",text)
  text=re.sub(r"\n{3,}","\n\n",text)
  text=text.strip()
  return text

In [23]:
cleaned_text=clean_text(sample["text"])
print(cleaned_text[:5000])

Unknown Car Loan Terms & Conditions
Terms and Conditions
New Car Loan Terms and Conditions
1. I agree to abide by the Bank's Terms and Conditions and rules in force and the changes thereto in
Terms and Conditions from time to time relating to my account as communicated and made available
on the Bank's website.
2. I agree that the opening and maintenance of the account is subject to rules and regulations
introduced or amended from time to time by the Reserve Bank of India.
3. I agree that the bank before opening any deposit account, will carry out a due diligence as required
under Know Your Customer guidelines of the bank. I would be required to submit necessary documents
or proofs, such as identity , address, photograph and any such information to meet with KYC, AML or
other statutory/regulatory requirements.
Further, after the account is opened, in compliance with the extant regulatory guidelines, I agree to
submit the above documents again at periodic intervals, as may be required by

In [24]:
def fix_encoding(text):
    """
    Fix common encoding artifacts found in PDFs.
    """

    replacements = {
        "ï¿½": "'",
        "â€™": "'",
        "â€œ": '"',
        "â€": '"',
        "â€“": "-",
        "â€”": "-",
        "Â": "",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    return text

In [25]:
def normalize_whitespace(text):
    """
    Normalize spaces, tabs and newlines.
    """

    # Windows line endings
    text = text.replace("\r", "\n")

    # Tabs
    text = text.replace("\t", " ")

    # Multiple spaces
    text = re.sub(r" +", " ", text)

    # Three or more blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [26]:
def remove_ui_noise(text):
    """
    Remove common website UI elements.
    """

    noise_phrases = [
        "Accessibility Menu",
        "Color Adjustment",
        "Virtual Keyboard",
        "Hover on content",
        "Voice/Accent",
        "Page Structure",
        "Profiles",
        "Content",
        "Colors",
        "Volume",
        "Rate",
        "Pitch",
        "Apply Now",
        "opens in a new tab"
    ]

    for phrase in noise_phrases:
        text = text.replace(phrase, "")

    return text

In [27]:
def clean_text(text):
    """
    Master cleaning pipeline.
    """

    text = fix_encoding(text)

    text = normalize_whitespace(text)

    text = remove_ui_noise(text)

    # Normalize again because removals may create blank spaces
    text = normalize_whitespace(text)

    return text

In [28]:
cleaned_text = clean_text(sample["text"])

print(cleaned_text[:5000])

Unknown Car Loan Terms & Conditions
Terms and Conditions
New Car Loan Terms and Conditions
1. I agree to abide by the Bank's Terms and Conditions and rules in force and the changes thereto in
Terms and Conditions from time to time relating to my account as communicated and made available
on the Bank's website.
2. I agree that the opening and maintenance of the account is subject to rules and regulations
introduced or amended from time to time by the Reserve Bank of India.
3. I agree that the bank before opening any deposit account, will carry out a due diligence as required
under Know Your Customer guidelines of the bank. I would be required to submit necessary documents
or proofs, such as identity , address, photograph and any such information to meet with KYC, AML or
other statutory/regulatory requirements.
Further, after the account is opened, in compliance with the extant regulatory guidelines, I agree to
submit the above documents again at periodic intervals, as may be required by

## Testing Utilities

The following helper functions are used to inspect and validate the PDF processing pipeline during development.

These functions are intended only for testing and debugging.

In [29]:
def list_documents():
    """
    Display all discovered PDF documents with their index.
    """

    print("=" * 90)
    print("AVAILABLE PDF DOCUMENTS")
    print("=" * 90)

    for index, pdf in enumerate(pdf_files):
        print(f"[{index:2}] {pdf.relative_to(DATASET_DIR)}")

In [30]:
list_documents()

AVAILABLE PDF DOCUMENTS
[ 0] HDFC/Car Loan/HDFC_Car_Loan_MITC.pdf
[ 1] HDFC/Car Loan/HDFC_Car_Loan_Overview.pdf
[ 2] HDFC/Home Loan/HDFC_Home_Loan_MITC.pdf
[ 3] HDFC/Home Loan/HDFC_Home_Loan_Overview.pdf
[ 4] HDFC/Personal Loan/HDFC_Personal_Loan_Overview.pdf
[ 5] SBI/Education Loan/SBI_EDUCATION_LOAN_Main_Page.pdf
[ 6] SBI/Education Loan/SBI_EDUCATION_LOAN_Scholar_Loan..pdf
[ 7] SBI/Education Loan/SBI_EDUCATION_LOAN_Student_Loan.pdf
[ 8] SBI/Education Loan/SBI_Education_Loan_Interest_Rate.pdf
[ 9] SBI/Education Loan/SBI_Education_MITC_Scholar_Loan.pdf
[10] SBI/Education Loan/SBI_Educational_Loan_Faq's.pdf
[11] SBI/Gold Loan/SBI_GOLD_LOAN_Product_Overview.pdf.pdf
[12] SBI/Home Loan/SBI_Home_Loan_Agreement.pdf
[13] SBI/Home Loan/SBI_Home_Loan_Application_Form.pdf
[14] SBI/Home Loan/SBI_Home_Loan_Documents_Required.pdf
[15] SBI/Home Loan/SBI_Home_Loan_EMI_Calculator.pdf
[16] SBI/Home Loan/SBI_Home_Loan_Eligibility.pdf
[17] SBI/Home Loan/SBI_Home_Loan_FAQs.pdf
[18] SBI/Home Loan/SBI_Home_

In [31]:
def preview_cleaning(index, preview_chars=3000):
    """
    Preview cleaned text for a selected PDF.
    """

    pdf_path = pdf_files[index]

    print("=" * 90)
    print(f"Document : {pdf_path.relative_to(DATASET_DIR)}")
    print("=" * 90)

    document = extract_text_from_pdf(pdf_path)

    cleaned_text = clean_text(document["text"])

    print(cleaned_text[:preview_chars])

In [32]:
preview_cleaning(0)

Document : HDFC/Car Loan/HDFC_Car_Loan_MITC.pdf
Unknown Car Loan Terms & Conditions
Terms and Conditions
New Car Loan Terms and Conditions
1. I agree to abide by the Bank's Terms and Conditions and rules in force and the changes thereto in
Terms and Conditions from time to time relating to my account as communicated and made available
on the Bank's website.
2. I agree that the opening and maintenance of the account is subject to rules and regulations
introduced or amended from time to time by the Reserve Bank of India.
3. I agree that the bank before opening any deposit account, will carry out a due diligence as required
under Know Your Customer guidelines of the bank. I would be required to submit necessary documents
or proofs, such as identity , address, photograph and any such information to meet with KYC, AML or
other statutory/regulatory requirements.
Further, after the account is opened, in compliance with the extant regulatory guidelines, I agree to
submit the above documents ag

In [33]:
def preview_metadata(index):
    """
    Display extracted metadata for a selected PDF.
    """

    pdf_path = pdf_files[index]

    metadata = extract_metadata_from_path(pdf_path)

    print(json.dumps(metadata, indent=4))

In [34]:
preview_metadata(0)

{
    "bank": "HDFC",
    "loan_type": "Car Loan",
    "file_name": "HDFC_Car_Loan_MITC.pdf",
    "file_path": "HDFC/Car Loan/HDFC_Car_Loan_MITC.pdf",
    "extension": ".pdf",
    "processed_date": "2026-07-08"
}


In [35]:
def preview_statistics(index):
    """
    Display statistics for a selected PDF.
    """

    pdf_path = pdf_files[index]

    document = extract_text_from_pdf(pdf_path)

    print("=" * 50)
    print("DOCUMENT STATISTICS")
    print("=" * 50)

    print(f"Pages      : {document['page_count']}")
    print(f"Characters : {document['character_count']:,}")
    print(f"Words      : {document['word_count']:,}")

In [36]:
preview_statistics(0)

DOCUMENT STATISTICS
Pages      : 6
Characters : 19,927
Words      : 3,277


In [37]:
def detect_document_type(filename):
    """
    Detect the document type from the filename.
    """

    filename = filename.lower()

    if "faq" in filename:
        return "FAQ"

    elif "mitc" in filename:
        return "MITC"

    elif "term" in filename:
        return "Terms & Conditions"

    elif "condition" in filename:
        return "Terms & Conditions"

    elif "interest" in filename:
        return "Interest Rates"

    else:
        return "Overview"

In [38]:
def extract_metadata_from_path(pdf_path):

    relative_path = pdf_path.relative_to(DATASET_DIR)

    parts = relative_path.parts

    metadata = {

        "bank": parts[0],

        "loan_type": parts[1],

        "document_type": detect_document_type(pdf_path.name),

        "file_name": pdf_path.name,

        "file_path": str(relative_path),

        "extension": pdf_path.suffix,

        "processed_date": datetime.now().strftime("%Y-%m-%d")

    }

    return metadata

In [39]:
preview_metadata(0)

{
    "bank": "HDFC",
    "loan_type": "Car Loan",
    "document_type": "MITC",
    "file_name": "HDFC_Car_Loan_MITC.pdf",
    "file_path": "HDFC/Car Loan/HDFC_Car_Loan_MITC.pdf",
    "extension": ".pdf",
    "processed_date": "2026-07-08"
}


In [41]:
def process_pdf(pdf_path):
  """Complete processing pipeline for a single PDF."""
  metadata=extract_metadata_from_path(pdf_path)
  extracted=extract_text_from_pdf(pdf_path)
  cleaned=clean_text(extracted["text"])
  document={"metadata":metadata,"statistics":{"page_count":extracted["page_count"],"character_count":extracted["character_count"],"word_count":extracted["word_count"]}, "content":{"raw_text":extracted["text"],"cleaned_text":cleaned}}
  return document

In [42]:
document=process_pdf(pdf_files[0])

In [43]:
document.keys()

dict_keys(['metadata', 'statistics', 'content'])

In [50]:
def save_document(document):
  """Save cleaned text and metadata while preserving the dataset folder structure."""
  metadata=document["metadata"]
  bank=metadata["bank"]
  loan=metadata["loan_type"]
  text_folder=CLEAN_TEXT_DIR/bank/loan
  meta_folder=METADATA_DIR/bank/loan
  text_folder.mkdir(parents=True,exist_ok=True)
  meta_folder.mkdir(parents=True,exist_ok=True)
  filename=Path(metadata["file_path"]).stem
  text_file=text_folder/f"{filename}.txt"
  with open(text_file,"w",encoding="utf-8") as f:
    f.write(document["content"]["cleaned_text"])
  metadata_file=meta_folder/f"{filename}.json"
  with open(metadata_file,"w",encoding='utf-8') as f:
    metadata_to_save = {
    **document["metadata"],
    **document["statistics"]
}

    json.dump(metadata_to_save, f, indent=4)
  return text_file,metadata_file

In [51]:
document=process_pdf(pdf_files[0])
text_file,metadata_file=save_document(document)
print(text_file)
print(metadata_file)

/content/drive/MyDrive/AI_Loan_Advisory_Chatbot/processed/cleaned_text/HDFC/Car Loan/HDFC_Car_Loan_MITC.txt
/content/drive/MyDrive/AI_Loan_Advisory_Chatbot/processed/metadata/HDFC/Car Loan/HDFC_Car_Loan_MITC.json


In [52]:
processed_documents=[]
for pdf in pdf_files:
  document=process_pdf(pdf)
  save_document(document)
  processed_documents.append(document)
print(f"\nSuccessfully processed {len(processed_documents)} documents")


Successfully processed 36 documents


In [53]:
total_pages = 0
total_words = 0
total_characters = 0

for doc in processed_documents:

    total_pages += doc["statistics"]["page_count"]

    total_words += doc["statistics"]["word_count"]

    total_characters += doc["statistics"]["character_count"]

print("="*60)
print("PDF PROCESSING SUMMARY")
print("="*60)

print(f"Documents Processed : {len(processed_documents)}")
print(f"Total Pages         : {total_pages}")
print(f"Total Words         : {total_words:,}")
print(f"Total Characters    : {total_characters:,}")

print("="*60)

PDF PROCESSING SUMMARY
Documents Processed : 36
Total Pages         : 250
Total Words         : 55,956
Total Characters    : 352,534


In [54]:
processed_documents = []
failed_documents = []

for pdf in pdf_files:

    try:
        document = process_pdf(pdf)

        save_document(document)

        processed_documents.append(document)

        print(f"✅ Processed: {pdf.name}")

    except Exception as e:

        failed_documents.append({
            "file": str(pdf.relative_to(DATASET_DIR)),
            "error": str(e)
        })

        print(f"❌ Failed: {pdf.name}")
        print(f"   Error: {e}")

✅ Processed: HDFC_Car_Loan_MITC.pdf
✅ Processed: HDFC_Car_Loan_Overview.pdf
✅ Processed: HDFC_Home_Loan_MITC.pdf
✅ Processed: HDFC_Home_Loan_Overview.pdf
✅ Processed: HDFC_Personal_Loan_Overview.pdf
✅ Processed: SBI_EDUCATION_LOAN_Main_Page.pdf
✅ Processed: SBI_EDUCATION_LOAN_Scholar_Loan..pdf
✅ Processed: SBI_EDUCATION_LOAN_Student_Loan.pdf
✅ Processed: SBI_Education_Loan_Interest_Rate.pdf
✅ Processed: SBI_Education_MITC_Scholar_Loan.pdf
✅ Processed: SBI_Educational_Loan_Faq's.pdf
✅ Processed: SBI_GOLD_LOAN_Product_Overview.pdf.pdf
✅ Processed: SBI_Home_Loan_Agreement.pdf
✅ Processed: SBI_Home_Loan_Application_Form.pdf
✅ Processed: SBI_Home_Loan_Documents_Required.pdf
✅ Processed: SBI_Home_Loan_EMI_Calculator.pdf
✅ Processed: SBI_Home_Loan_Eligibility.pdf
✅ Processed: SBI_Home_Loan_FAQs.pdf
✅ Processed: SBI_Home_Loan_Features.pdf
✅ Processed: SBI_Home_Loan_Interest_Rates.pdf
✅ Processed: SBI_Home_Loan_Interest_Rates_and_Fees.pdf
✅ Processed: SBI_Home_Loan_Process_Guide.pdf
✅ Processed

In [55]:
log_file = OUTPUTS_DIR / "processing_log.txt"

with open(log_file, "w", encoding="utf-8") as f:

    f.write("AI Loan Advisory Chatbot\n")
    f.write("PDF Processing Log\n")
    f.write("=" * 60 + "\n\n")

    f.write(f"Successful Documents : {len(processed_documents)}\n")
    f.write(f"Failed Documents     : {len(failed_documents)}\n\n")

    if failed_documents:

        f.write("FAILED FILES\n")
        f.write("-" * 60 + "\n")

        for item in failed_documents:

            f.write(f"{item['file']}\n")

            f.write(f"Error : {item['error']}\n\n")

print("Processing log saved successfully.")

Processing log saved successfully.


In [56]:
total_pages = sum(doc["statistics"]["page_count"] for doc in processed_documents)

total_words = sum(doc["statistics"]["word_count"] for doc in processed_documents)

total_characters = sum(doc["statistics"]["character_count"] for doc in processed_documents)

print("\n" + "=" * 70)
print("AI LOAN ADVISORY CHATBOT - PDF PROCESSING SUMMARY")
print("=" * 70)

print(f"Total PDFs Found          : {len(pdf_files)}")
print(f"Successfully Processed    : {len(processed_documents)}")
print(f"Failed                    : {len(failed_documents)}")

print("-" * 70)

print(f"Total Pages               : {total_pages}")
print(f"Total Words               : {total_words:,}")
print(f"Total Characters          : {total_characters:,}")

print("-" * 70)

if len(failed_documents) == 0:
    print("Status                    : SUCCESS")
else:
    print("Status                    : COMPLETED WITH WARNINGS")

print("=" * 70)


AI LOAN ADVISORY CHATBOT - PDF PROCESSING SUMMARY
Total PDFs Found          : 36
Successfully Processed    : 36
Failed                    : 0
----------------------------------------------------------------------
Total Pages               : 250
Total Words               : 55,956
Total Characters          : 352,534
----------------------------------------------------------------------
Status                    : SUCCESS


In [57]:
required_keys = {"metadata", "statistics", "content"}

print("Running validation...\n")

for i, document in enumerate(processed_documents):

    missing = required_keys - document.keys()

    if missing:
        print(f"❌ Document {i} missing keys: {missing}")
    else:
        print(f"✅ Document {i} passed validation.")

print("\nValidation Complete.")

Running validation...

✅ Document 0 passed validation.
✅ Document 1 passed validation.
✅ Document 2 passed validation.
✅ Document 3 passed validation.
✅ Document 4 passed validation.
✅ Document 5 passed validation.
✅ Document 6 passed validation.
✅ Document 7 passed validation.
✅ Document 8 passed validation.
✅ Document 9 passed validation.
✅ Document 10 passed validation.
✅ Document 11 passed validation.
✅ Document 12 passed validation.
✅ Document 13 passed validation.
✅ Document 14 passed validation.
✅ Document 15 passed validation.
✅ Document 16 passed validation.
✅ Document 17 passed validation.
✅ Document 18 passed validation.
✅ Document 19 passed validation.
✅ Document 20 passed validation.
✅ Document 21 passed validation.
✅ Document 22 passed validation.
✅ Document 23 passed validation.
✅ Document 24 passed validation.
✅ Document 25 passed validation.
✅ Document 26 passed validation.
✅ Document 27 passed validation.
✅ Document 28 passed validation.
✅ Document 29 passed validatio